# Experiment 04 — Fine-Tune M500 on Dark-Skin Melanoma (StyleGAN2-ADA)

**Goal:** Fine-tune `M500.pkl` on the curated **139 dark-skin dermoscopic ISIC 2020 melanoma images** (`ITA <= 28°`) using StyleGAN2-ADA (`NVlabs/stylegan3`), then measure whether the generated skin-tone distribution shifts toward darker skin tones.

**Requirements:**
- Colab **T4 GPU** runtime (`Runtime -> Change runtime type -> T4 GPU`)
- Colab Secrets: `DAGSHUB_USERNAME` and `DAGSHUB_TOKEN` enabled

In [ ]:
# 1. Verify GPU runtime
import os, sys, subprocess
from pathlib import Path
import torch

assert torch.cuda.is_available(), 'No GPU detected! Switch Colab Runtime to T4 GPU.'
print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 2. Clone StyleGAN3 (with StyleGAN2-ADA training support) & project repo
SG3_DIR = Path('/content/stylegan3')
if not SG3_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/NVlabs/stylegan3.git', str(SG3_DIR)], check=True)
sys.path.insert(0, str(SG3_DIR))

REPO_ROOT = Path('/content/bias-aware-stylegan')
if not REPO_ROOT.exists():
    subprocess.run([
        'git', 'clone', '-b', 'experiment/m500-baseline',
        'https://github.com/Muneeb-Ahmad-404/bias-aware-stylegan.git', str(REPO_ROOT)
    ], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', 'origin', 'experiment/m500-baseline'], check=True)

os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print('Working dir:', Path.cwd())

In [ ]:
# 3. Install dependencies & pull M500.pkl via DVC
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'dvc', 'ninja', 'pyspng', 'scipy', 'opencv-python', 'pandas', 'matplotlib', 'kagglehub', 'pyyaml', 'click', 'tqdm', 'psutil'
], check=True)

from google.colab import userdata
dagshub_user = userdata.get('DAGSHUB_USERNAME')
dagshub_token = userdata.get('DAGSHUB_TOKEN')
assert dagshub_user and dagshub_token, 'Set DAGSHUB_USERNAME and DAGSHUB_TOKEN in Colab Secrets!'

subprocess.run([sys.executable, '-m', 'dvc', 'remote', 'modify', '--local', 'origin_processed', 'auth', 'basic'], check=True)
subprocess.run([sys.executable, '-m', 'dvc', 'remote', 'modify', '--local', 'origin_processed', 'user', dagshub_user], check=True)
subprocess.run([sys.executable, '-m', 'dvc', 'remote', 'modify', '--local', 'origin_processed', 'password', dagshub_token], check=True)
subprocess.run([sys.executable, '-m', 'dvc', 'pull', 'models/pretrained/M500.pkl.dvc'], check=True)

In [ ]:
# 4. Download & curate the 139 dark-skin ISIC 2020 melanoma images, then pack into StyleGAN zip
subprocess.run([sys.executable, 'src/data/ingest.py'], check=True)
subprocess.run([sys.executable, '-m', 'src.data.curate_dark_skin', '--ita-threshold', '28.0'], check=True)

DATASET_ZIP = Path('/content/dark_isic_ita28.zip')
if not DATASET_ZIP.exists():
    subprocess.run([
        sys.executable, str(SG3_DIR / 'dataset_tool.py'),
        '--source=data/processed/finetune_isic_ita28',
        f'--dest={DATASET_ZIP}',
        '--resolution=512x512'
    ], check=True)
print('Dataset ready:', DATASET_ZIP, DATASET_ZIP.stat().st_size // (1024 * 1024), 'MB')

In [ ]:
# 5. Fine-tune M500.pkl with StyleGAN2-ADA (~20-35 min for 40 kimg on T4)
# Checkpoints are saved every 2 ticks (~8 kimg) into /content/training-runs/
OUT_DIR = Path('/content/training-runs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, str(SG3_DIR / 'train.py'),
    f'--outdir={OUT_DIR}',
    '--cfg=stylegan2',
    f'--data={DATASET_ZIP}',
    '--gpus=1',
    '--batch=8',
    '--gamma=5',
    '--mirror=1',
    '--aug=ada',
    '--target=0.6',
    '--resume=models/pretrained/M500.pkl',
    '--kimg=40',
    '--snap=2',
    '--metrics=none',
    '--workers=2',
]
print('Running:', ' '.join(cmd))
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
ret = proc.wait()
if ret != 0:
    raise RuntimeError(f'train.py exited with code {ret}')

In [ ]:
# 6. Locate latest fine-tuned checkpoint and generate 500 images (seeds 0..499)
import legacy
import numpy as np
from PIL import Image
from tqdm import tqdm
import torch_utils.ops.bias_act as bias_act
import torch_utils.ops.upfirdn2d as upfirdn2d

_orig_bias_act = bias_act.bias_act
bias_act.bias_act = lambda *a, **kw: _orig_bias_act(*a, **{**kw, 'impl': 'ref'})
_orig_upfirdn2d = upfirdn2d.upfirdn2d
upfirdn2d.upfirdn2d = lambda *a, **kw: _orig_upfirdn2d(*a, **{**kw, 'impl': 'ref'})

ckpts = sorted(OUT_DIR.rglob('network-snapshot-*.pkl'))
assert ckpts, 'No checkpoints found in /content/training-runs!'
best_ckpt = ckpts[-1]
print('Using checkpoint:', best_ckpt)

device = torch.device('cuda')
with open(best_ckpt, 'rb') as f:
    G_ema = legacy.load_network_pkl(f)['G_ema'].to(device).eval()
for p in G_ema.parameters():
    p.requires_grad_(False)

GEN_DIR = Path('/content/finetuned_samples_500')
GEN_DIR.mkdir(parents=True, exist_ok=True)

batch_size = 10
seeds = list(range(500))
c_dummy = torch.zeros([batch_size, G_ema.c_dim], device=device)

with torch.no_grad():
    for i in tqdm(range(0, len(seeds), batch_size), desc='Generating'):
        batch_seeds = seeds[i:i + batch_size]
        zs = torch.stack([
            torch.from_numpy(np.random.RandomState(s).randn(G_ema.z_dim)).float()
            for s in batch_seeds
        ]).to(device)
        imgs = G_ema(zs, c_dummy[:len(batch_seeds)], truncation_psi=0.7, noise_mode='const')
        imgs = (imgs.permute(0, 2, 3, 1) * 127.5 + 128).clamp(0, 255).to(torch.uint8).cpu().numpy()
        for s, arr in zip(batch_seeds, imgs):
            Image.fromarray(arr, 'RGB').save(GEN_DIR / f'seed_{s:04d}.png')

print('Generated:', len(list(GEN_DIR.glob('*.png'))), 'images')

In [ ]:
# 7. Run ITA skin-tone analysis & compare against M500 baseline
import json
import matplotlib.pyplot as plt
from src.evaluation.ita import ITA_BINS, analyze_image_folder, save_ita_report

df_ft, summary_ft = analyze_image_folder(GEN_DIR)
save_ita_report(df_ft, summary_ft, 'reports/finetune_v1', prefix='ita')
print(json.dumps(summary_ft, indent=2))

# Plot distribution + sample 4x4 grid
order = [b[0] for b in ITA_BINS]
pcts_ft = [summary_ft['distribution'][g]['percentage'] for g in order]
baseline_pcts = [37.4, 24.6, 15.0, 12.8, 7.0, 3.2]

x = range(len(order))
width = 0.38
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar([i - width/2 for i in x], baseline_pcts, width, label='M500 Baseline', color='#B4C7E7')
ax.bar([i + width/2 for i in x], pcts_ft, width, label='M500 Fine-Tuned (Dark ISIC)', color='#7F6000')
ax.set_xticks(list(x))
ax.set_xticklabels(order, rotation=20)
ax.set_ylabel('Percentage (%)')
ax.set_title('Skin-Tone Shift: M500 Baseline vs Fine-Tuned (N=500)')
ax.legend()
plt.tight_layout()
plt.savefig('reports/finetune_v1/baseline_vs_finetuned_ita.png', dpi=150)
plt.show()

# 4x4 visual grid of generated samples
sample_imgs = sorted(GEN_DIR.glob('*.png'))[:16]
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
for ax, p in zip(axes.flatten(), sample_imgs):
    ax.imshow(Image.open(p))
    ax.set_title(p.stem, fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.savefig('reports/finetune_v1/sample_grid.png', dpi=150)
plt.show()